# ZORE AI — Kantor Eksekutif 3D di Blender

Notebook ini membuat aset kantor 3D lengkap: ruang marmer, dinding panel walnut dan aksen brass, meja direktur, ruang rapat dengan kursi menghadap meja, workstation, partisi, tanaman, dan karakter karyawan sederhana. Hasil disimpan ke Google Drive.

Setelah Drive terhubung, jalankan satu-satunya sel kode dengan tombol ▶. Tunggu sampai muncul `SELESAI`. Proses instalasi Blender memerlukan beberapa menit.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, textwrap
WORK_DIR='/content/drive/MyDrive/ZORE-AI/Blender'
os.makedirs(WORK_DIR,exist_ok=True)
print('1/4 Drive tersambung:',WORK_DIR,flush=True)
print('2/4 Memasang Blender...',flush=True)
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','blender'],check=True)
subprocess.run(['blender','--version'],check=True)
print('3/4 Membuat kantor eksekutif 3D ZORE AI...',flush=True)
blender_script = "import bpy, math, os\nfrom mathutils import Vector\n\nOUT=os.environ[\"ZORE_OUTPUT_DIR\"]\nos.makedirs(OUT, exist_ok=True)\nbpy.ops.object.select_all(action=\"SELECT\"); bpy.ops.object.delete(use_global=False)\nfor datablocks in (bpy.data.materials, bpy.data.curves, bpy.data.cameras, bpy.data.lights):\n    pass\n\ndef material(name, color, metallic=0.0, roughness=0.55):\n    m=bpy.data.materials.new(name); m.diffuse_color=(*color,1); m.use_nodes=True\n    bs=m.node_tree.nodes.get(\"Principled BSDF\")\n    bs.inputs[\"Base Color\"].default_value=(*color,1)\n    bs.inputs[\"Metallic\"].default_value=metallic\n    bs.inputs[\"Roughness\"].default_value=roughness\n    return m\n\nivory=material(\"Warm ivory plaster\",(0.72,.66,.55),0,.82)\ncream=material(\"Ceiling cream\",(.86,.81,.70),0,.8)\nwalnut=material(\"Dark walnut wood\",(.19,.085,.045),0,.38)\nwalnut2=material(\"Walnut highlights\",(.31,.16,.085),0,.42)\nbrass=material(\"Brushed champagne brass\",(.58,.39,.16),.72,.28)\nmarble=material(\"Cream marble\",(.68,.64,.56),.08,.28)\ngreen=material(\"Deep emerald upholstery\",(.035,.19,.135),0,.72)\nleather=material(\"Cognac leather\",(.30,.12,.055),0,.62)\nblack=material(\"Graphite metal\",(.035,.045,.05),.65,.35)\nglass=material(\"Smoked blue glass\",(.12,.23,.27),.35,.18)\nskin=material(\"Character skin\",(.58,.34,.22),0,.78)\nnavy=material(\"Employee navy fabric\",(.045,.075,.12),0,.82)\nsage=material(\"Employee sage fabric\",(.16,.27,.20),0,.82)\nwhite=material(\"Paper and ceramic\",(.82,.80,.72),0,.5)\n\ndef bevel(obj, amount=.06, segments=3):\n    mod=obj.modifiers.new(\"Soft crafted edges\",\"BEVEL\"); mod.width=amount; mod.segments=segments\n    obj.modifiers.new(\"Weighted corner normals\",\"WEIGHTED_NORMAL\")\n    return obj\n\ndef cube(name, loc, scale, mat, bevel_amt=.035):\n    bpy.ops.mesh.primitive_cube_add(size=1, location=loc); o=bpy.context.object; o.name=name\n    o.dimensions=scale; bpy.ops.object.transform_apply(location=False, rotation=False, scale=True)\n    if mat:o.data.materials.append(mat)\n    if bevel_amt:bevel(o,min(bevel_amt,min(scale)*.22))\n    return o\n\ndef cyl(name, loc, radius, depth, mat, vertices=32, rotation=None):\n    bpy.ops.mesh.primitive_cylinder_add(vertices=vertices, radius=radius, depth=depth, location=loc, rotation=rotation or (0,0,0))\n    o=bpy.context.object;o.name=name\n    if mat:o.data.materials.append(mat)\n    bevel(o,.018,2);return o\n\ndef sphere(name, loc, scale, mat):\n    bpy.ops.mesh.primitive_uv_sphere_add(segments=20, ring_count=12, location=loc);o=bpy.context.object;o.name=name;o.scale=scale\n    bpy.ops.object.transform_apply(location=False, rotation=False, scale=True)\n    if mat:o.data.materials.append(mat)\n    return o\n\ndef text_obj(body, loc, size, mat, extrude=.008):\n    cu=bpy.data.curves.new(\"Engraved lettering\",\"FONT\");cu.body=body;cu.size=size;cu.extrude=extrude;cu.align_x=\"CENTER\"\n    o=bpy.data.objects.new(body,cu);bpy.context.collection.objects.link(o);o.location=loc\n    if mat:o.data.materials.append(mat)\n    return o\n\n# Architectural shell: open-front executive office for clear viewing.\ncube(\"Polished marble floor\",(0,0,-.18),(24,.32,18),marble,.08)\ncube(\"Rear plaster wall\",(0,8.85,4.1),(24,.35,8.5),ivory,.04)\ncube(\"Left return wall\",(-11.85,0,4.1),(.3,17.7,8.5),ivory,.04)\ncube(\"Right return wall\",(11.85,0,4.1),(.3,17.7,8.5),ivory,.04)\ncube(\"Ceiling floating cornice\",(0,8.55,8.15),(23.8,.55,.22),brass,.035)\n# Walnut panel wall with repeated brass reveals.\nfor x in [i*1.65-10.7 for i in range(14)]:\n    cube(\"Full height walnut wall panel\",(x,8.62,4.1),(1.48,.10,7.6),walnut,.025)\n    cube(\"Panel brass reveal\",(x+.77,8.54,4.1),(.035,.035,7.35),brass,.008)\ncube(\"Feature wall inset\",(0,8.48,4.35),(8.2,.10,4.7),walnut2,.04)\ncube(\"ZORE plaque backing\",(0,8.38,5.45),(5.6,.10,1.15),brass,.035)\ncube(\"ZORE plaque dark inset\",(0,8.30,5.45),(5.35,.08,.92),walnut,.025)\n# Text faces toward the room (-Y).\nlogo=text_obj(\"ZORE AI\",(0,8.20,5.27),.62,brass,.012);logo.rotation_euler[0]=math.radians(90)\nsub=text_obj(\"EXECUTIVE WORKSPACE\",(0,8.20,4.83),.16,cream,.004);sub.rotation_euler[0]=math.radians(90)\n# Tall side windows and frames.\nfor x in [-9.7,-6.8,6.8,9.7]:\n    cube(\"Panoramic window\",(x,8.55,4.4),(2.25,.06,4.7),glass,.02)\n    for dx in [-1.16,1.16]:cube(\"Window brass stile\",(x+dx,8.48,4.4),(.055,.08,4.85),brass,.01)\n    cube(\"Window sill\",(x,8.43,2.0),(2.4,.22,.10),brass,.015)\n# Executive desk in the rear half.\ncube(\"Executive desk walnut top\",(0,3.2,1.55),(5.8,2.0,.22),walnut,.09)\ncube(\"Executive desk brass reveal\",(0,3.2,1.42),(5.82,2.02,.055),brass,.018)\nfor x in [-2.15,2.15]:\n    cube(\"Executive desk pedestal\",(x,3.2,.78),(.72,1.55,1.3),walnut2,.07)\ncube(\"Desk leather writing pad\",(0,3.05,1.69),(3.0,1.15,.035),green,.02)\ncube(\"Desk monitor\",(0,4.0,2.35),(1.65,.10,1.05),black,.045)\ncube(\"Monitor glass\",(0,3.94,2.36),(1.48,.025,.86),glass,.018)\ncube(\"Monitor stand\",(0,4.0,1.82),(.18,.16,.42),brass,.025)\n# Desk accessories.\ncube(\"Desk document tray\",(-1.85,2.85,1.76),(.85,.55,.09),black,.025)\nfor i in range(3):cube(\"Stacked presentation folders\",(-1.85,2.85,1.83+i*.045),(.7,.42,.035),white,.008)\ncyl(\"Desk pen holder\",(1.9,3.1,1.86),.13,.30,brass)\n# Executive chair behind desk, facing desk/front of room.\ncube(\"Executive chair seat\",(0,5.0,1.15),(1.25,1.15,.28),leather,.12)\ncube(\"Executive chair back\",(0,5.55,2.0),(1.25,.28,1.65),leather,.12)\ncyl(\"Executive chair column\",(0,5.0,.62),.10,.78,brass)\nfor a in range(5):\n    ang=a*math.tau/5\n    cube(\"Executive chair star base\",(math.cos(ang)*.48,5.0+math.sin(ang)*.48,.20),(.72,.10,.09),black,.035)\n# Meeting table in foreground; chairs face toward table.\ncube(\"Conference table walnut slab\",(0,-3.0,1.32),(7.6,2.0,.20),walnut,.10)\ncube(\"Conference table brass edge\",(0,-3.0,1.20),(7.62,2.02,.055),brass,.018)\nfor x in [-2.8,2.8]:cube(\"Conference table pedestal\",(x,-3.0,.62),(.45,1.45,1.15),black,.06)\ncube(\"Conference table runner\",(0,-3.0,1.44),(3.8,.36,.035),green,.02)\n# Meeting chairs on both long sides, seats/back orientation toward table.\nfor x in [-2.7,-.9,.9,2.7]:\n    # front row faces +Y\n    cube(\"Meeting chair seat\",(x,-4.75,.82),(.95,.95,.20),green,.10)\n    cube(\"Meeting chair back\",(x,-5.18,1.42),(.95,.20,1.05),green,.10)\n    cyl(\"Meeting chair pedestal\",(x,-4.75,.40),.075,.65,brass)\n    # far row faces -Y\n    cube(\"Meeting chair seat\",(x,-1.25,.82),(.95,.95,.20),green,.10)\n    cube(\"Meeting chair back\",(x,-.82,1.42),(.95,.20,1.05),green,.10)\n    cyl(\"Meeting chair pedestal\",(x,-1.25,.40),.075,.65,brass)\n# Side workstations with task chairs.\nfor x in [-8.2,8.2]:\n    cube(\"Side workstation desk\",(x,1.1,1.25),(3.0,1.35,.16),walnut2,.06)\n    for dx in [-1.05,1.05]:cube(\"Workstation leg\",(x+dx,1.1,.62),(.12,1.05,1.1),black,.02)\n    cube(\"Workstation screen\",(x,1.55,2.0),(1.1,.10,.72),black,.035)\n    cube(\"Workstation display\",(x,1.49,2.0),(.96,.025,.58),glass,.01)\n# Decorative divider with vertical slats.\ncube(\"Timber divider base\",(-5.4,-1.2,1.0),(.18,5.3,2.0),walnut,.04)\nfor i in range(13):\n    y=-3.65+i*.41\n    cube(\"Divider vertical slat\",(-5.4,y,2.25),(.11,.12,2.5),walnut2,.025)\ncube(\"Divider brass cap\",(-5.4,-1.2,3.55),(.18,5.35,.09),brass,.02)\n# Columns, ceiling details, recessed lights.\nfor x in [-10,10]:\n    cyl(\"Decorative brass column\",(x,5.4,3.8),.13,7.5,brass)\n    cyl(\"Column capital\",(x,5.4,7.55),.27,.20,walnut2)\nfor x in [-8,-4,0,4,8]:\n    cube(\"Ceiling recessed light\",(x,1.0,7.9),(1.5,.65,.08),brass,.04)\n    cube(\"Ceiling diffuser\",(x,1.0,7.84),(1.28,.48,.04),cream,.025)\n# Indoor planters with leaves.\nfor x,y in [(-9,-5),(9,-5),(-9,5),(9,5)]:\n    cyl(\"Planter pot\",(x,y,.42),.48,.82,walnut2,24)\n    cyl(\"Planter brass rim\",(x,y,.82),.50,.09,brass,24)\n    for j in range(7):\n        a=j*math.tau/7\n        leaf=sphere(\"Broad indoor plant leaf\",(x+math.cos(a)*.36,y+math.sin(a)*.36,1.45),(.18,.12,.78),sage)\n        leaf.rotation_euler[1]=math.cos(a)*.35;leaf.rotation_euler[0]=math.sin(a)*.35\n    cyl(\"Plant central stem\",(x,y,1.25),.055,.9,sage,12)\n# Stylized 3D employees: torso, head, hair, arms and legs.\ndef employee(name,x,y,cloth,face_y=0):\n    # seated/standing figure assembled as separate meshes, facing into work zone\n    sphere(name+\" torso\",(x,1.35,y),(.38,.62,.25),cloth)\n    sphere(name+\" head\",(x,2.18,y),(.29,.34,.29),skin)\n    sphere(name+\" hair\",(x,2.40,y),(.30,.18,.30),walnut)\n    # neck\n    cyl(name+\" neck\",(x,1.88,y),.12,.22,skin)\n    # arms angled down to desk\n    for side in [-1,1]:\n        arm=cube(name+\" arm\",(x+side*.43,1.52,y-.02),(.19,.72,.20),cloth,.07)\n        arm.rotation_euler[2]=side*.20\n        leg=cube(name+\" leg\",(x+side*.16,.48,y),(.22,.82,.25),navy,.07)\n    return\nemployee(\"ZORE employee one\",-2.2,5.0,navy)\nemployee(\"ZORE employee two\",-7.6,2.0,sage)\nemployee(\"ZORE employee three\",7.6,2.0,navy)\n# A few sculptural objects.\nfor x in [-1.5,1.5]:\n    cyl(\"Table centerpiece bowl\",(x,-3,1.56),.22,.10,brass)\n# Camera and soft studio lights are included in the blend file for future editing.\nbpy.ops.object.camera_add(location=(20,-27,19))\ncam=bpy.context.object;cam.name=\"ZORE Office Presentation Camera\"\ndirection=Vector((0,0,2.4))-cam.location;cam.rotation_euler=direction.to_track_quat(\"-Z\",\"Y\").to_euler();cam.data.lens=38\nbpy.context.scene.camera=cam\nfor name,loc,power,size in [(\"Large softbox key\",(-8,-5,12),2600,9),(\"Warm wall wash\",(2,7,10),1800,7),(\"Meeting fill\",(9,-1,9),1500,6)]:\n    bpy.ops.object.light_add(type=\"AREA\", location=loc);l=bpy.context.object;l.name=name;l.data.energy=power;l.data.shape=\"DISK\";l.data.size=size;l.rotation_euler=(Vector((0,0,2.5))-l.location).to_track_quat(\"-Z\",\"Y\").to_euler()\n# Organize blend settings.\nscene=bpy.context.scene\nscene.render.engine=\"BLENDER_EEVEE_NEXT\"\nscene.render.resolution_x=1600;scene.render.resolution_y=1000;scene.render.resolution_percentage=100\nscene.world.color=(.22,.22,.22)\nscene.view_settings.view_transform=\"AgX\"\n# Save native project and export complete scene as GLB.\nblend_path=os.path.join(OUT,\"zore-ai-executive-office.blend\")\nglb_path=os.path.join(OUT,\"zore-ai-executive-office.glb\")\nbpy.ops.wm.save_as_mainfile(filepath=blend_path)\nbpy.ops.export_scene.gltf(filepath=glb_path,export_format=\"GLB\",export_apply=True)\nprint(\"ZORE_OFFICE_EXPORT_OK\")\n"
script_path='/content/zore_office_build.py'
with open(script_path,'w',encoding='utf-8') as fp: fp.write(blender_script)
env=os.environ.copy();env['ZORE_OUTPUT_DIR']=WORK_DIR
run=subprocess.run(['blender','-b','-t','2','--python',script_path],env=env,capture_output=True,text=True)
print(run.stdout[-5000:],flush=True)
if run.returncode:
 print(run.stderr[-5000:],flush=True)
 raise RuntimeError('Pembuatan kantor gagal. Kirim screenshot error merah.')
print('4/4 Memeriksa hasil...',flush=True)
for name in ['zore-ai-executive-office.blend','zore-ai-executive-office.glb']:
 path=os.path.join(WORK_DIR,name)
 if not os.path.isfile(path) or os.path.getsize(path)==0: raise RuntimeError('File hasil tidak ditemukan: '+name)
 print(name,os.path.getsize(path),'bytes',flush=True)
print('SELESAI — folder Google Drive / ZORE-AI / Blender',flush=True)


## Hasil yang diharapkan

- `zore-ai-executive-office.blend` — proyek Blender yang dapat diedit.
- `zore-ai-executive-office.glb` — model untuk dimuat ke preview ZORE AI.

Di halaman preview tekan **Muat model Blender**, lalu pilih file GLB tersebut. Ini adalah aset 3D tahap awal yang bisa diperiksa; karakter masih berupa figur 3D sederhana, belum rig animasi atau AI yang bergerak. Notebook ini belum menghubungkan antrean tugas karyawan secara otomatis.